# 05.03 · Structured outputs

**Chapter:** Working with LLM APIs · **Milestone:** M2

## Goals
- [ ] Why "please answer in JSON" isn't enough
- [ ] Define schemas with Pydantic; validate everything that comes back
- [ ] Three approaches: native structured outputs (`response_format`), Instructor, constrained decoding (Outlines / local engines)
- [ ] Measure extraction accuracy on a small labelled set

## Resources
- [OpenAI – Structured outputs guide](https://platform.openai.com/docs/guides/structured-outputs) — free
- [Instructor docs](https://python.useinstructor.com/) · [Outlines docs](https://dottxt-ai.github.io/outlines/) — free
- [Pydantic docs](https://docs.pydantic.dev/) — free

LLM output is text; your program needs data. Structured outputs are how LLMs plug into real software: extraction, classification,
routing, tool arguments, UI rendering.

| Approach | How | Use when |
|---|---|---|
| Prompt + `json.loads` | ask nicely, parse, retry | never in production on its own |
| JSON mode | model guaranteed to emit valid JSON (any shape) | schema too dynamic for a fixed model |
| **Native structured outputs** | provider constrains decoding to your JSON schema | default when the provider supports it |
| **Instructor** | Pydantic + automatic validation + re-asking with the error | many providers, custom validators |
| **Constrained decoding** (Outlines, llama.cpp grammars, vLLM) | engine masks tokens that would break the schema | local / self-hosted models |

In [ ]:
import json
from datetime import date
from typing import Literal

from pydantic import BaseModel, Field, ValidationError, field_validator

from aieng.llm import client

c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}

## 1. The naive way and why it breaks
Models add prose ("Sure! Here's the JSON:"), markdown fences, trailing commas, wrong keys, wrong types.

In [ ]:
r = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content":
    "Extract name and age as JSON from: 'Hi, I'm Carlos and I turned 31 last week.'"}], **EXTRA)
raw = r.choices[0].message.content
print(repr(raw))
try:
    print(json.loads(raw))
except json.JSONDecodeError as e:
    print("json.loads failed:", e)

## 2. Pydantic schemas
A Pydantic model is the single source of truth: it generates the JSON schema sent to the model **and** validates the response.
`Field(description=...)` text is sent to the model too — it's part of your prompt.

In [ ]:
class LineItem(BaseModel):
    description: str
    quantity: int = Field(ge=1)
    unit_price: float = Field(ge=0, description="price of one unit, no currency symbol")


class Invoice(BaseModel):
    vendor: str
    invoice_date: date = Field(description="ISO date YYYY-MM-DD")
    currency: Literal["USD", "EUR", "PEN", "GBP"]
    items: list[LineItem]
    total: float

    @field_validator("total")
    @classmethod
    def total_matches_items(cls, v, info):
        items = info.data.get("items") or []
        expected = round(sum(i.quantity * i.unit_price for i in items), 2)
        if items and abs(expected - v) > 0.05:
            raise ValueError(f"total {v} != sum of items {expected}")
        return v


print(json.dumps(Invoice.model_json_schema(), indent=1)[:600], "...")

## 3. Native structured outputs
`client.chat.completions.parse(..., response_format=Invoice)` sends the schema and parses into the Pydantic object.
Supported by OpenAI, Gemini (OpenAI-compatible endpoint), Ollama, vLLM and others — check your provider.

In [ ]:
text = """INVOICE — Café Andino SAC — 2026-03-14
2 x Espresso beans 1kg ...... 45.00 each
1 x Grinder burr set ........ 120.50
Total due: PEN 210.50"""

r = c.chat.completions.parse(model=MODEL, messages=[
    {"role": "system", "content": "Extract the invoice. Use only information present in the text."},
    {"role": "user", "content": text}], response_format=Invoice, temperature=0, **EXTRA)
inv = r.choices[0].message.parsed
print(inv)

Constrained decoding guarantees the **shape**, not the **truth**: the model can still extract wrong numbers.
And custom validators (our `total` check) run *after* parsing — `parse` raises if they fail.

## 4. Instructor: validation + automatic retries
Instructor wraps any client. If validation fails, it re-asks the model with the error message, up to `max_retries`.

In [ ]:
import instructor

ic = instructor.from_openai(c, mode=instructor.Mode.JSON)
inv2 = ic.chat.completions.create(model=MODEL, response_model=Invoice, max_retries=3, temperature=0, **EXTRA,
                                  messages=[{"role": "user", "content": f"Extract the invoice:\n{text}"}])
print(inv2.model_dump())

## 5. Classification with `Literal` / enums
Structured outputs make classifiers trivial and parse-proof.

In [ ]:
class Ticket(BaseModel):
    category: Literal["billing", "bug", "feature_request", "account", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(description="max 12 words")


for msg in ["I was charged twice this month!!", "Dark mode would be cool someday", "App crashes when I upload a PDF, I have a demo in 1 hour"]:
    t = c.chat.completions.parse(model=MODEL, messages=[{"role": "user", "content": msg}], response_format=Ticket, temperature=0, **EXTRA)
    print(t.choices[0].message.parsed)

## 6. Measure, don't eyeball
A tiny labelled set and an accuracy number beat ten impressive demos. (Chapter 10 makes this a habit.)

In [ ]:
labelled = [
    ("ACME Corp, 2026-01-05. 3 x Widget @ 10.00. Total USD 30.00", {"vendor": "ACME Corp", "total": 30.00, "n_items": 1}),
    ("Bill from Panadería Rosa (2026-02-11): 12 x pan francés 0.50, 2 x torta 35.00 — total PEN 76.00", {"vendor": "Panadería Rosa", "total": 76.00, "n_items": 2}),
    ("Receipt | Bücherwelt GmbH | 2026-04-02 | 1 Buch 24.90 EUR | 2 Lesezeichen à 2.50 | Summe 29.90 EUR", {"vendor": "Bücherwelt GmbH", "total": 29.90, "n_items": 2}),
    ("London Cab Co. 2026-05-20. One airport transfer, 1 x 65.00 GBP. Total GBP 65.00", {"vendor": "London Cab Co.", "total": 65.00, "n_items": 1}),
]

def score(pred: Invoice | None, gold):
    if pred is None:
        return 0.0
    checks = [pred.vendor.strip().lower() == gold["vendor"].lower(), abs(pred.total - gold["total"]) < 0.01, len(pred.items) == gold["n_items"]]
    return sum(checks) / len(checks)

scores = []
for text_i, gold in labelled:
    try:
        pred = ic.chat.completions.create(model=MODEL, response_model=Invoice, max_retries=2, temperature=0, **EXTRA,
                                          messages=[{"role": "user", "content": f"Extract the invoice:\n{text_i}"}])
    except Exception as e:  # retries exhausted
        print("failed:", type(e).__name__)
        pred = None
    scores.append(score(pred, gold))
    print(f"{scores[-1]:.2f}  {pred.vendor if pred else None!r} total={pred.total if pred else None}")
print(f"field accuracy: {sum(scores) / len(scores):.0%} with {MODEL}")

## Exercises
1. Write 10 messy invoices (different languages, formats, missing fields) with gold labels. Measure field-level accuracy with two models.
2. Make `invoice_date` optional (`date | None`) and instruct the model to return null when absent. Does it hallucinate dates without that?
3. `uv add outlines` and generate a `Ticket` with constrained decoding on a local Hugging Face model (SmolLM2 or Qwen3-0.6B).

In [ ]:
# your code here